In [1]:
import os
import glob
path = "../data"
import pandas as pd
xlsx_pattern = os.path.join(path, "*.xlsx")
all_files = glob.glob(xlsx_pattern)

In [2]:
li = [pd.read_excel(file_name) for file_name in all_files]
df = pd.concat(li, axis=0, ignore_index=True)
df

,year,month,date,day of week,member_A_s,member_A_e,member_B_s,member_B_e,member_C_s,member_C_e,member_D_s,member_D_e,member_E_s,member_E_e,member_F_s,member_F_e,member_G_s,member_G_e
0,2024,12,1,NaN,NaN,NaN,15.0,23.0,NaN,NaN,NaN,NaN,7.0,15.0,NaN,NaN,23.0,7.0
1,2024,12,2,NaN,9.0,17.0,NaN,NaN,7.0,15.0,16.0,23.0,NaN,NaN,NaN,NaN,23.0,7.0
2,2024,12,3,NaN,9.0,17.0,7.0,15.0,NaN,NaN,16.0,23.0,NaN,NaN,23.0,9.0,NaN,NaN
3,2024,12,4,NaN,9.0,17.0,NaN,NaN,9.0,15.0,NaN,NaN,16.0,23.0,23.0,9.0,NaN,NaN
4,2024,12,5,NaN,9.0,17.0,NaN,NaN,9.0,1530.0,NaN,NaN,16.0,23.0,23.0,9.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,2025,1,27,1.0,9.0,17.0,NaN,NaN,16.0,23.0,7,15,NaN,NaN,NaN,NaN,23.0,7.0
58,2025,1,28,2.0,9.0,17.0,7.0,15.0,10.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,2025,1,29,3.0,9.0,17.0,17.0,23.0,NaN,NaN,12,17,NaN,NaN,23.0,7.0,NaN,NaN
60,2025,1,30,4.0,9.0,17.0,17.0,23.0,NaN,NaN,12,17,NaN,NaN,23.0,7.0,NaN,NaN


In [3]:
df = df.rename(columns={'date': 'day'})
df['time'] = pd.to_datetime(df[['year', 'month', 'day']])
df = df.drop(columns=['year', 'month', 'day','day of week'])
df.head()

,member_A_s,member_A_e,member_B_s,member_B_e,member_C_s,member_C_e,member_D_s,member_D_e,member_E_s,member_E_e,member_F_s,member_F_e,member_G_s,member_G_e,time
0,NaN,NaN,15.0,23.0,NaN,NaN,NaN,NaN,7.0,15.0,NaN,NaN,23.0,7.0,2024-12-01
1,9.0,17.0,NaN,NaN,7.0,15.0,16.0,23.0,NaN,NaN,NaN,NaN,23.0,7.0,2024-12-02
2,9.0,17.0,7.0,15.0,NaN,NaN,16.0,23.0,NaN,NaN,23.0,9.0,NaN,NaN,2024-12-03
3,9.0,17.0,NaN,NaN,9.0,15.0,NaN,NaN,16.0,23.0,23.0,9.0,NaN,NaN,2024-12-04
4,9.0,17.0,NaN,NaN,9.0,1530.0,NaN,NaN,16.0,23.0,23.0,9.0,NaN,NaN,2024-12-05


In [15]:
def caculate_shift_distinct(df):
    # ---------------------------
    # (2) Identify employee names
    # ---------------------------

    columns = df.columns.tolist()
    employees = sorted(set(col[:-2] for col in columns if col.endswith(("_s", "_e"))))
    distinct_shifts = []

    for emp in employees:
        start_col = emp + "_s"
        end_col   = emp + "_e"
        valid_rows = df.dropna(subset=[start_col, end_col])
        distinct_pairs = valid_rows[[start_col, end_col]]
        distinct_pairs = distinct_pairs.rename(columns={start_col: 'start',end_col: 'end'})
        distinct_shifts.append(distinct_pairs)
    df_all_shifts = pd.concat(distinct_shifts, axis=0, ignore_index=True)
    return df_all_shifts.reset_index(drop=True).drop_duplicates()
distinct_shifts = caculate_shift_distinct(df)
distinct_shifts


,start,end
0,9.0,17.0
23,9.0,15.0
45,15.0,23.0
46,7.0,15.0
54,17.0,23.0
60,16.0,23.0
80,9.0,1530.0
112,10.0,16.0
133,x,x
136,,
